# 04 — Model Training: model × imbalance-strategy grid

**Models compared and why:**

| Model | Why it's here | Limitation |
|---|---|---|
| Logistic Regression | Fast, interpretable linear baseline — the bar to beat | Linear boundaries only |
| Random Forest | Robust non-linear bagging benchmark | Large, slower inference |
| XGBoost | Boosting, state of the art on tabular data | More hyperparameters |
| LightGBM | Boosting, fastest training | Sensitive to leaf settings on small minority |
| Balanced Random Forest | Undersamples majority *inside each* bootstrap — imbalance-native | Each tree sees less data |

**Imbalance strategies compared:** `class_weight` (reweight the loss), `SMOTE` (synthesize minority
neighbors), `undersample` (drop majority rows), `SMOTE+Tomek` (oversample then clean boundary pairs).
All resampling lives **inside** imblearn pipelines → resampling happens only on fit data, never on
validation/test.

**Selection metric: PR-AUC** (average precision) on the validation set — at 0.17% positives, ROC-AUC
is inflated by the enormous true-negative pool, while PR-AUC directly reflects the precision/recall
trade-off analysts and customers experience.

The full grid runs via `python -m src.train_pipeline` (results cached to `artifacts/experiments.csv`).
Re-running the grid here would take ~15 min; the cell below loads the cached results if present.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display
from src.utils import load_config, resolve_path

config = load_config()
pd.set_option("display.max_columns", 40)

In [ ]:
results_path = resolve_path(config["paths"]["artifacts_dir"]) / "experiments.csv"
if results_path.exists():
    results = pd.read_csv(results_path)
else:                                # full re-run (slow) — only if no cache
    from src.data_loader import load_raw_data
    from src.preprocessing import drop_duplicates, make_splits
    from src.feature_engineering import engineer_features, get_feature_columns
    from src.model_training import run_experiment_grid
    df = engineer_features(drop_duplicates(load_raw_data(config)), config)
    splits = make_splits(df, get_feature_columns(df, config), config)
    results = run_experiment_grid(splits, config)
results.style.background_gradient(subset=["val_pr_auc"], cmap="Greens")

### Reading the table

- **Class weighting beats resampling** for the strong learners: gradient boosting gets its minority
  signal from loss weighting without inventing synthetic points.
- **SMOTE** is competitive but adds training cost, and its synthetic frauds are interpolations —
  patterns real fraud need not follow.
- **Random undersampling** consistently loses: it throws away ~99.8% of genuine rows and with them
  the shape of the majority distribution.
- **ROC-AUC barely separates the rows while PR-AUC does** — exactly why we select on PR-AUC.

The winner is then hyperparameter-tuned with `RandomizedSearchCV` (stratified 3-fold, average
precision objective); the tuned model is kept only if it beats the untuned validation score.

In [ ]:
from src.utils import load_json
meta = load_json(resolve_path(config["paths"]["artifacts_dir"]) / "model_metadata.json")
{k: meta[k] for k in ("model", "strategy", "tuned", "val_pr_auc") if k in meta}